In [1]:
# === IMPORT LIBRARIES ===
import os
import sys
import subprocess
import ctypes
# from unsloth import FastLanguageModel

# === CUDA SYSTEM BOOT FIX ===

# Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# === FILEPATH SETUP ===

# 1. Inject Codebase into Python Path (Wipe cache first for Jupyter safety)
import sys
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

codebase_path = "../"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)
print(f"✅ Codebase mounted at: {codebase_path}")

# 2. Configure Global Filepaths
CACHE_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"


✅ Codebase mounted at: ../


In [3]:
# === IMPORT HF API KEY ===
from huggingface_hub import login

# Load HF token from artifacts/.env
env_path = ENV_PATH
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            if line.strip() and not line.startswith("#") and "=" in line:
                key, val = line.strip().split("=", 1)
                if key.strip() == "HF_TOKEN":
                    login(token=val.strip())
                    print("Successfully logged into Hugging Face Hub!")
                    break
else:
    print(f"Warning: {env_path} not found.")

Successfully logged into Hugging Face Hub!


In [4]:
# === IMPORT LIBRARIES ===
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from transformers.training_args import TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from src.utils.prompts import format_prompt

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
# === CONFIGURATION ===
# MODEL_ID = "Qwen/Qwen3-30B-A3B-Instruct-2507"
MODEL_ID = "ibm-granite/granite-3.1-2b-instruct" # ANCHOR: MODEL_ID configured for Unslot usage.\n

OUTPUT_DIR = f"{MODELS_DIR}/sft_structOnly_granite-3.1-2b-instruct" 
TRAIN_DATA_PATH = f"{CACHE_DIR}/train_structural.jsonl" # train_full_info or train_structural
VAL_DATA_PATH = f"{CACHE_DIR}/val_structural.jsonl" # val_full_info or val_structural

In [6]:
# === LOAD DATA ===
dataset = load_dataset("text", data_files={"train": TRAIN_DATA_PATH, "val": VAL_DATA_PATH})

# Apply formatting
import json
def apply_chat_template(example, tokenizer):
    example = json.loads(example['text'])
    messages = format_prompt(example)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {'text': prompt}


In [7]:
# === LOAD MODEL & TOKENIZER ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

processed_dataset = dataset.map(lambda x: apply_chat_template(x, tokenizer), num_proc=8)

# Compute data type: bfloat16 for local server
COMPUTE_DTYPE = torch.bfloat16

print(f"Targeting compute data type: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    # device_map="cuda:0",
    device_map="auto",
    trust_remote_code=True,
    dtype=COMPUTE_DTYPE,
)

# === TEST: LOADING Qwen3-30B MoE WITH UNSLOTH ===
# max_memory override is REQUIRED: accelerate uses BF16 weight sizes (~57GB) for
# device placement BEFORE quantization. Without this, it offloads layers to CPU,
# which bitsandbytes forbids for 4-bit QLoRA training.
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_ID,
#     max_seq_length=2048,
#     dtype=COMPUTE_DTYPE,
#     load_in_4bit=True,
#     max_memory={0: "80GiB"},
# )

Targeting compute data type: torch.bfloat16


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

### Parameter Breakdown: LoRA Config (PiSSA and DoRA is also supported)

*   **`r` (Rank) [TWEAKABLE]:** Determines the size of the low-rank adaptation matrices. A higher rank (e.g., 32, 64) allows the model to learn more complex patterns but increases trainable parameters and VRAM. A lower rank (e.g., 8, 16) is faster and prevents overfitting.
*   **`lora_alpha` [TWEAKABLE]:** The scaling factor for LoRA weights. Generally, set this to `2 * r` (e.g., if `r=16`, `lora_alpha=32`). Higher alpha weights the LoRA updates more heavily.
*   **`lora_dropout` [TWEAKABLE]:** The dropout probability for LoRA layers (typically 0.05 to 0.1) to reduce overfitting.
*   **`target_modules` [DO NOT TOUCH]:** Specifies which modules inside the model's architecture should receive LoRA adapters. The setting below targeting attention and MLP projections (`q_proj`, `k_proj`, etc.) is standard for model fine-tuning and covers the entire Transformer block.

In [8]:
# === LoRA CONFIG ===
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    
    # === TRYING OUT DIFFERENT FINE-TUNING METHODS ===
    # init_lora_weights="pissa_niter_16" # ANCHOR: Enables PiSSA (Principal Singular values and Singular vectors Adaptation). Significantly 
	# use_dora=True # ANCHOR: Enables DoRA (Weight-Decomposed Low-Rank Adaptation)
	# Comment out this section just to run LoRA.
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 28,180,480 || all params: 2,561,712,128 || trainable%: 1.1001


### Parameter Breakdown: TrainingArguments

*   **`per_device_train_batch_size` [TWEAKABLE]:** The batch size per GPU. If you experience Out of Memory (OOM) errors, lower this to 1. If you have plenty of VRAM, you can raise it (e.g., 4 or 8) to speed up training.
*   **`gradient_accumulation_steps` [TWEAKABLE]:** Dictates how many steps to accumulate gradients before performing a backward pass. This simulates a larger batch size without increasing memory requirements. **Effective Batch Size = `per_device_train_batch_size` * `gradient_accumulation_steps`**. (E.g., 2 * 4 = 8).
*   **`learning_rate` [TWEAKABLE]:** Adjusts step size during optimization. Standard SFT learning rates range from `5e-5` to `2e-4`. Too high can cause training instability; too low makes training progress extremely slow.
*   **`fp16` or `bf16` [TWEAKABLE]:** Use `fp16=True` for older GPUs. If your GPU supports bfloat16 (Ampere architectures and newer, like A100, RTX 3090/4090), **change this to `bf16=True`** for better numerical stability.
*   **`optim` [DO NOT TOUCH]:** `paged_adamw_8bit` is highly recommended. It offloads memory-intensive optimizer states to the CPU pages if the GPU runs out of VRAM, serving as a critical safety net against OOM errors.

In [9]:
# === TRAINING CONFIGURATION ===
training_args = SFTConfig(
	output_dir=OUTPUT_DIR,
	per_device_train_batch_size=1,
	gradient_accumulation_steps=4,
	learning_rate=2e-4,
	logging_steps=10,
	num_train_epochs=1,
	eval_strategy="steps",
	eval_steps=100,
	save_strategy="steps",
	save_steps=100,
	bf16=True,
	fp16=False,
	optim="paged_adamw_8bit",
	dataset_text_field="text",
	max_length=1500,
	
	# === MEMORY OPTIMIZATIONS FOR 32B MODEL ===
	# per_device_eval_batch_size=1,      # Reduce eval batch size from default 8 to 1
	# gradient_checkpointing=True,       # Enable gradient checkpointing to save VRAM
	# eval_accumulation_steps=1,         # Prevent accumulating eval tensors on GPU
)

### Parameter Breakdown: SFTTrainer

*   **`max_seq_length` [TWEAKABLE]:** The maximum context length (in tokens) the model is allowed to read. Setting this too low will truncate your inputs (context + steps + options), while setting it too high will exponentially increase VRAM usage. 2048 is a solid baseline for tool-calling datasets. If your EDA indicates contexts are longer, you can try 3072 or 4096 (but monitor GPU usage closely).

In [10]:
# === SFT TRAINER ===

trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset['train'],
    eval_dataset=processed_dataset['val'],
    processing_class=tokenizer,
    args=training_args,
)

# To resume from checkpoint, run: trainer.train(resume_from_checkpoint=True)
trainer.train()
trainer.save_model(OUTPUT_DIR)

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.347542,0.351981,0.353955,492857.000000,0.910285
200,0.299589,0.279555,0.290152,985338.000000,0.927239
300,0.255550,0.239856,0.257781,1478734.000000,0.937271
400,0.229236,0.216658,0.227535,1973917.000000,0.943640
500,0.187996,0.200535,0.207895,2470864.000000,0.947499
600,0.197402,0.187606,0.195109,2968956.000000,0.950808
700,0.189617,0.180961,0.193092,3464824.000000,0.952411
800,0.174918,0.173192,0.180975,3958835.000000,0.954382
900,0.168363,0.169892,0.177467,4447292.000000,0.955135
1000,0.164422,0.164095,0.173838,4936884.000000,0.956646
